# Simple MLP on MNIST

A baseline dense network (no spiking) trained on MNIST using the SpikeNet `DataLoader` wrapper. Useful as a reference before comparing against spiking variants.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as transforms

from spikenet.dataloader import DataLoader
from spikenet.network import Network

## Data

Subclass `DataLoader` to wrap MNIST. `x_transform` flattens images from `(1, 28, 28)` to `(784,)` — the shape expected by a dense network.

In [ ]:
class MyData(DataLoader):
    def __init__(self):
        super().__init__(
            train_data=torchvision.datasets.MNIST(
                root="./~data/mnist",
                train=True,
                transform=transforms.ToTensor(),
                download=True,
            ),
            test_data=torchvision.datasets.MNIST(root="./~data/mnist", train=False, transform=transforms.ToTensor()),
        )

    def x_transform(self, x):
        # Transform data into 1D array
        return x.reshape(-1, 28 * 28)


data = MyData()

Visualise a random training sample to verify the data and transform are working correctly.

In [ ]:
x, y = data.sample()
img = x.reshape(28, 28).numpy()
plt.imshow(img, cmap="gray")
plt.title(f"This is a: {y.item()}")
plt.show()

## Network

Build a 2-layer MLP: 784 → 500 → 10. `add_layer` accepts any `torch.nn.Module` — standard PyTorch layers, spiking layers, or custom modules. The network stays uncompiled until `.fit()` is called, at which point input shape is inferred from the dataloader.

In [ ]:
net = (
    Network("Simple Network")
    .add_layer(torch.nn.Linear, 500)
    .add_layer(torch.nn.ReLU, in_features=500)
    .add_layer(torch.nn.Linear, 10)
)

net.summary()

## Training

Call `.fit()` to compile the network, run the training loop, and evaluate on the test set. The compiled structure is shown in the updated summary.

In [ ]:
net = net.fit_on(data)
net.summary()